In [7]:
# pip install langgraph

In [ ]:
'''.
AI Agents -->
Agentic AI --> Before taking action it thinks first
TypedDict --> To get the output in Dict Format.
''' 

'.\nAI Agents -->\nAgentic AI --> Before taking action it thinks first\nTypedDict --> To get the output in Dict Format.\n'

In [ ]:
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END 
from typing import TypedDict
from langchain_core.tools import tool
import os

In [10]:
gemini_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = os.environ['Gemini_api'],
    temperature = 0
)

qwen_model = ChatOllama(
    model = 'qwen2.5:1.5b',
    temperature=0
)

In [11]:
@tool
def calculator(expression : str) -> str:
    """Calculate a mathematical expression."""
    try:
        result = eval(expression)
        return str(result)
    except Exception:
        return "Invalid mathematical expression"

In [ ]:
class AgentState(TypedDict): # TypedDict --> To get the Output in Dict format
    question : str
    decision : str
    tool_result: str
    answer : str 

def agent(state : AgentState):
    question = state['question']
    prompt = f''' 
    You 're an AI Agent.
    User Question : {question}

    Decide what to do.

    If the questions requires calculation,
    reponse exactly : USE_TOOL

    otherwise respond exactly : DIRECT_ANSWER
    '''

    response = gemini_model.invoke(prompt)
    decision = response.content.strip()
    return {"decision" : decision}

def use_tool(state : AgentState):
    question = state['question']
    prompt = f"""
    Extract only the mathematical expression
    from this question.
    
    Question:
    {question}
    
    Example:
    "What is 25 * 10?"
    Return:
    25 * 10
    """

    response = gemini_model.invoke(prompt)

    expression = response.content.strip()

    result = calculator.invoke({
        "expression" : expression
    })
    return {"tool_result" : result}

def final_answer(state : AgentState):
    question = state['question']
    tool_result = state.get("tool_result","") # if the key called tool_result it takes that if not empty string not 'None'
    if tool_result:
        prompt = f"""
        Answer the user's question.

        Question:
        {question}

        Calculator Result:
        {tool_result}

        Give only the final answer.
        """

    else:
        prompt = f"""
        Answer this question clearly:

        {question}
        """

    response = gemini_model.invoke(prompt)

    return {"answer" : response.content}

def router(state : AgentState):
    decision = state['decision']

    if 'USE_TOOL' in decision:
        return 'tool'
    return 'answer'


graph = StateGraph(AgentState)

graph.add_node("agent", agent)
graph.add_node("tool", use_tool)
graph.add_node("answer", final_answer)

graph.add_edge(START, "agent")

graph.add_conditional_edges(
    "agent",
    router,
    {
        "tool": "tool",
        "answer": "answer"
    }
)

graph.add_edge("tool", "answer")
graph.add_edge("answer", END)

app = graph.compile()

result = app.invoke({
    "question": "What is 125 * 24?",
    "decision": "",
    "tool_result": "",
    "answer": ""
})


print("\nAgent Answer:")
print(result["answer"])

In [ ]:
# nodes --> adding functionalities